In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from index_evaluator import load_authors_data, evaluate_all_with_expert_index
from data_utils import load_author_publications
from index_metrics import (
    h_index, g_index, i10_index, ha_index, ar_index,
    timed_h_index, contemporary_h_index, trend_h_index,
    career_years_h_index_by_average_citations_per_year,
)
from viz_utils import *

In [ ]:
# 설정
authors_file = "./gsc_data/authors.all"
data_dir = "./gsc_data/DATA/"
output_file = "all_index_results.csv"
current_year = 2013

In [ ]:
# 1. 지수 계산 및 로딩
print("📥 지수 계산 또는 로드 중...")
authors_data = load_authors_data(authors_file)
if not os.path.exists(output_file):
    index_df, max_q, max_r = evaluate_all_with_expert_index(authors_data, data_dir, current_year)
    print(max_q, max_r)
    index_df.to_csv(output_file, index=False)
else:
    index_df = pd.read_csv(output_file)

In [ ]:
start_year = 1970
end_year = 2013

authors_data = load_authors_data(authors_file)

for year in range(start_year, end_year + 1):
    expert_by_year_file = f"index_by_{year}.csv"
    if not os.path.exists(expert_by_year_file):
        print(f"📆 전체 author 연도별 지수 계산 중...{expert_by_year_file}")
        index_df, max_q, max_r = evaluate_all_with_expert_index(authors_data, data_dir, year)
        print(max_q, max_r)
        index_df.to_csv(expert_by_year_file, index=False)
        print(f"{year}년도 계산 완료")
    else:
        index_df = pd.read_csv(expert_by_year_file)
        # print(f"{year}년도 로드 완료")
        pass
        # df_expert_by_year = pd.read_csv(expert_by_year_file)
        # print("📂 기존 expert_index_by_year.csv 로드 완료")



In [ ]:
# 2. 컬럼 이름 축약 및 분석 대상 설정
index_df_short = index_df.rename(columns=INDEX_NAME_MAP)
index_columns = list(INDEX_NAME_MAP.values())

In [ ]:
# 3. 분석 및 시각화 실행
print("📊 상관관계 분석 중...")
# plot_index_scatter(index_df, "expert_index", "h_index")
# plot_index_scatter(index_df, "expert_index", "g_index")
# plot_index_scatter(index_df, "expert_index", "i10_index")
# plot_index_scatter(index_df, "expert_index", "contemporary_h_index")
# plot_index_scatter(index_df, "expert_index", "career_year_h_index_by_publications")
# plot_index_scatter(index_df, "expert_index", "career_year_h_index_by_publications_year_citations")
# plot_index_scatter(index_df, "expert_index", "career_years_h_index_by_average_citations_per_year")
# plot_index_scatter(index_df, "expert_index", "timed_h_index_5")
# plot_index_scatter(index_df, "expert_index", "timed_h_index_10")
# plot_index_scatter(index_df, "expert_index", "ha_index")


In [ ]:
corr_pearson = plot_index_correlation_heatmap(index_df, method='pearson')
print(corr_pearson)
corr_spearman = plot_index_correlation_heatmap(index_df, method='spearman')
print(corr_spearman)

In [ ]:
print("🔝 Top-K 비교 및 겹침 분석 중...")
print_top_k_authors(index_df_short, k=1000, index_columns=index_columns)
overlap_matrix = compute_top_k_overlap_matrix(index_df_short, index_columns, k=1000)
# print(overlap_matrix)
plot_top_k_overlap_heatmap(overlap_matrix)

In [ ]:
print("🧍 독립 저자 분석 중...")
unique_authors_by_index = find_index_specific_top_k_authors(index_df_short, index_columns, k=20)
for idx, authors in unique_authors_by_index.items():
    print(f"🔹 {idx} 지수에만 Top-K로 포함된 저자 {len(authors)}명: {authors}")


In [ ]:
print("🔢 Top-K 확장에 따른 고유 저자 수 시각화 중...")
k_range = list(range(10, 110, 10))
author_counts_by_k = plot_unique_author_counts_by_k(index_df_short, index_columns, k_values=k_range)
print(author_counts_by_k)

In [ ]:
print("🔬 expert 고유 저자 특성 분석 중...")
unique_authors_by_index = find_index_specific_top_k_authors(index_df_short, index_columns, k=30)
expert_unique_authors = unique_authors_by_index.get("expert", set())
analyze_unique_authors_characteristics(expert_unique_authors, data_dir, current_year=current_year)


In [ ]:
print("🧭 지수 간 클러스터링 시각화 중...")
plot_index_clustermap(overlap_matrix, k=1000)

In [ ]:
print("📉 expert 기준 순위 급변 저자 분석 중...")
analyze_extreme_rank_shift(index_df_short, index_columns, base_index='expert', top_n=20)

In [ ]:
# 5. 특정 저자의 expert 시계열 추이 시각화 (정규화된 값 기준)
print("📊 특정 저자의 expert 시계열 분석 중...")
author_id = "nhL9PHwAAAAJ"
df_author_expert = df_expert_by_year[df_expert_by_year["author_id"] == author_id]

plt.figure(figsize=(8, 5))
plt.plot(df_author_expert["year"], df_author_expert["expert"], marker="o", label="expert")
plt.title(f"Expert Index Over Time: {author_id}")
plt.xlabel("Year")
plt.ylabel("Expert Score (normalized)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
print(df_author_expert)

In [ ]:
# 6. 전체 지수별 평균 시계열 분석 (추가 지수 포함)
print("📈 전체 지수별 연도별 평균 변화 분석 중...")
mean_df = df_expert_by_year.groupby("year").mean(numeric_only=True).reset_index()

plt.figure(figsize=(10, 6))
for col in mean_df.columns[1:]:
    plt.plot(mean_df["year"], mean_df[col], marker="o", label=col)
plt.title("Mean Index Trends Over Time (All Authors)")
plt.xlabel("Year")
plt.ylabel("Mean Score")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# author_id 지정
author_id = "nhL9PHwAAAAJ"
start_year = 1970
end_year = 2013
years = list(range(start_year, end_year + 1))

# expert 불러오기
df_expert = df_expert_by_year[df_expert_by_year["author_id"] == author_id].set_index("year")

# 저자 논문 불러오기
df_papers = load_author_publications(data_dir, author_id, current_year=end_year)

# 연도별 지수 계산
results = {
    "year": [],
    "h": [], "g": [], "i10": [], "ar": [], "ha": [],
    "h_t5": [], "h_t10": [], "hc": [], "ht": [],
    "cy_pub": [], "cy_cit": [], "cy_avg": [],
    "papers": [], "citations": [],
    "expert": []  # 병합할 예정
}

for year in years:
    papers = [(c, y) for c, y in zip(df_papers["citations"], df_papers["year"]) if y <= year]
    current_year_pubs = df_papers[df_papers["year"] == year]

    results["year"].append(year)
    results["h"].append(h_index(papers))
    results["g"].append(g_index(papers))
    results["i10"].append(i10_index(papers))
    results["ar"].append(ar_index(papers, year))
    results["ha"].append(ha_index(papers, year))
    results["h_t5"].append(timed_h_index(papers, year, 5))
    results["h_t10"].append(timed_h_index(papers, year, 10))
    results["hc"].append(contemporary_h_index(papers, year))
    results["ht"].append(trend_h_index(papers, year))
    results["cy_pub"].append(current_year_pubs.shape[0])
    results["cy_cit"].append(current_year_pubs["citations"].sum())
    results["cy_avg"].append(career_years_h_index_by_average_citations_per_year(papers))
    results["papers"].append(current_year_pubs.shape[0])
    results["citations"].append(current_year_pubs["citations"].sum())
    results["expert"].append(df_expert["expert"].get(year, 0))

# 시각화
df_author = pd.DataFrame(results)

# 라인 그래프
plt.figure(figsize=(12, 6))
for col in ["expert", "h", "g", "i10", "ar", "ha", "h_t5", "h_t10", "hc", "ht", "cy_avg"]:
    plt.plot(df_author["year"], df_author[col], marker="o", label=col)
plt.title(f"Index Trends for {author_id}")
plt.xlabel("Year")
plt.ylabel("Score")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True)
plt.tight_layout()
plt.show()

# 논문수/인용수 막대 그래프
df_author[["year", "citations"]].set_index("year").plot(kind="bar", figsize=(10,4), title="Papers & Citations per Year")
plt.ylabel("Count")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

# 논문수/인용수 막대 그래프
df_author[["year", "papers"]].set_index("year").plot(kind="bar", figsize=(10,4), title="Papers & Citations per Year")
plt.ylabel("Count")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

print(df_author)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# 연도별 지수 계산 결과로부터 DataFrame 생성
df_author = pd.DataFrame(results)

# Z-score 정규화 대상 지수 목록
index_cols = [
    "expert", "h", "g", "i10", "ar", "ha",
    "h_t5", "h_t10", "hc", "ht", "cy_avg"
]

# Z-score 정규화 적용
df_author_zscore = df_author.copy()
scaler = StandardScaler()
df_author_zscore[index_cols] = scaler.fit_transform(df_author[index_cols])



In [ ]:
plt.figure(figsize=(12, 6))
markers = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'H', 'X', '8']  # 최대 12개

for i, col in enumerate(index_cols):
    plt.plot(df_author_zscore["year"], df_author_zscore[col],
             marker=markers[i % len(markers)], label=col)

plt.title(f"Z-score Normalized Index Trends for {author_id}")
plt.xlabel("Year")
plt.ylabel("Z-score")
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
current_year = 2013
y = 2011

rec = max(round((1 - 0.1 * (current_year - 3 - y)), 2), 0.1)
val = max(round((1 - (current_year - 3 - y) * 0.1), 2), 0.1)

print(rec, val)